# M3 — Cluster-based recommender (behavioural)

This notebook implements a recommender that, given a user, returns the 5 product categories
most affine to their cluster. Recommendations are computed at cluster level, not at person level: all
members of the same segment receive the same top-5.

- User with history: their M1 behavioural segment (notebook `06`) is used, grouping users with
  similar click patterns.
- New user (cold start): their demographic cluster (notebook `05`) is used, which requires no history.

The recommender operates at cluster level because individual history is sparse (median ≈ 1 event per
user). In section 5 the weight `α` of the blend `α·individual + (1−α)·cluster` is tuned on validation
and the optimum turns out to be α = 0, so the final model serves the cluster top-5 directly with no
individual term.

## Outputs

- `data/processed/cluster_recommendations.csv` — top-5 per cluster.
- `data/processed/recommendations.csv` — top-5 per user, obtained via a `user → cluster` *lookup*
  to preserve the per-user interface of the API.

Evaluated without temporal leakage (section 5.bis), the cluster-based recommender does not beat the
popularity baseline: cluster HR@5 ≈ 0.59 versus popularity ≈ 0.63. The advantage observed previously
(HR@5 ≈ 0.69) stemmed from temporal leakage, since the `06` segment is computed using the entire history,
including the test period. The recommender retains operational value (interpretable recommendations that
are homogeneous per segment and valid in cold start), but it does not improve statistically on the
popularity baseline for predicting the next click.

## 0 · Libraries and paths

In [ ]:
import warnings
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.stats.contingency_tables import mcnemar
from scipy.stats import wilcoxon

# Leak-free segmentation pipeline (replica of 06): log1p -> scaler -> PCA -> KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

# Add src/ to the path and reuse the shared helper (same pattern as 01-04)
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(_root / "src"))
from tfm.utils import find_project_root  # noqa: E402

PROCESSED_PATH = find_project_root() / "data" / "processed"
np.random.seed(42)
print("Data folder:", PROCESSED_PATH)

## 1 · Load events, M1 segment (behavioural) and demographic cluster

In [ ]:
events = pd.read_csv(PROCESSED_PATH / "events.csv")
events["timestamp"] = pd.to_datetime(events["timestamp"])

# product_new (category) and sector — the sector is used to rebuild the leak-free segment (06)
products = pd.read_csv(PROCESSED_PATH / "products.csv")[["id_product", "product_new", "sector"]]
segmentos = pd.read_csv(PROCESSED_PATH / "users_segmented.csv")[["id_user", "segment"]]
clusters = pd.read_csv(PROCESSED_PATH / "users_demo_segments.csv")[["id_user", "demo_cluster"]]

# Clicks and opens only; target = 1 if there was a click
events = events[events["event_type"].isin(["click", "open"])].copy()
events["target"] = (events["event_type"] == "click").astype(int)

# Join: each event with its category, its sector, its segment and its cluster
df = events[["id_user", "id_product", "target", "timestamp"]].copy()
df = df.merge(products, on="id_product", how="left")
df = df.merge(segmentos, on="id_user", how="left")
df = df.merge(clusters, on="id_user", how="left")
df = df.dropna(subset=["product_new"]).reset_index(drop=True)

# List of all categories (in fixed order) and of the segments
ALL_CATS = sorted(df["product_new"].unique())
ALL_SEG = sorted(df["segment"].dropna().unique())
print("Events:", df.shape, "| categories:", len(ALL_CATS), "| segments:", len(ALL_SEG))

## 2 · Split into train / validation / test (by time)

- **train** (oldest 70 %): builds the signals.
- **validation** (10 %): selects the best α.
- **test** (most recent 20 %): the final evaluation.

In [ ]:
df = df.sort_values("timestamp").reset_index(drop=True)
corte_1 = int(len(df) * 0.70)
corte_2 = int(len(df) * 0.80)

train = df.iloc[:corte_1].copy()
valid = df.iloc[corte_1:corte_2].copy()
test = df.iloc[corte_2:].copy()

print("train:", len(train), "| valid:", len(valid), "| test:", len(test))
print("up to:", train["timestamp"].max().date(), "/", valid["timestamp"].max().date(), "/", test["timestamp"].max().date())

## 3 · Build the signals (using train data only)

- **afinidad_segmento:** click rate per (segment, category), normalised by row.
- **individual:** each user's click rate per category (their history).
- global **popularity** and **demographic cluster** (for baselines and cold start).

In [ ]:
def construir_senales(tr):
    # --- segment affinity: click rate of each segment in each category ---
    agg = tr.groupby(["segment", "product_new"])["target"].agg(n_eventos="count", n_clicks="sum")
    agg["tasa_click"] = agg["n_clicks"] / agg["n_eventos"]
    matriz = agg.reset_index().pivot(index="segment", columns="product_new", values="tasa_click")
    matriz = matriz.reindex(index=ALL_SEG, columns=ALL_CATS).fillna(0)
    # normalise each row so it sums to 1 (avoiding division by 0)
    suma_filas = matriz.sum(axis=1).replace(0, 1)
    afinidad_segmento = matriz.div(suma_filas, axis=0)

    # --- individual: each user's click rate per category ---
    individual = (tr.groupby(["id_user", "product_new"])["target"].mean()
                  .reset_index().pivot(index="id_user", columns="product_new", values="target"))

    # --- global popularity: number of clicks per category, normalised to [0, 1] ---
    conteo = tr[tr["target"] == 1]["product_new"].value_counts()
    popularidad = np.array([conteo.get(cat, 0) for cat in ALL_CATS], dtype=float)
    popularidad = popularidad / (popularidad.max() + 1e-9)

    # --- demographic cluster affinity ---
    conteo_cluster = (tr[tr["target"] == 1].groupby(["demo_cluster", "product_new"]).size()
                      .unstack(fill_value=0).reindex(columns=ALL_CATS, fill_value=0))
    cluster_aff = conteo_cluster.div(conteo_cluster.max(axis=1).replace(0, 1), axis=0)

    return afinidad_segmento, individual, popularidad, cluster_aff


afinidad_segmento, individual, popularidad, cluster_aff = construir_senales(train)

# Maps user -> segment and user -> cluster (for fast lookup)
usuario_segmento = df.drop_duplicates("id_user").set_index("id_user")["segment"]
usuario_cluster = df.drop_duplicates("id_user").set_index("id_user")["demo_cluster"]
print("Signals built. Segment affinity:", afinidad_segmento.shape)

## 4 · Functions for scoring and for measuring

In [ ]:
def matriz_segmento(user_ids, afinidad_seg, popularidad):
    """For each user, the affinity row of their segment (or popularity if they have no segment)."""
    filas = []
    for uid in user_ids:
        seg = usuario_segmento.get(uid)
        if seg in afinidad_seg.index:
            filas.append(afinidad_seg.loc[seg].values)
        else:
            filas.append(popularidad)
    return np.array(filas)


def matriz_hibrida(user_ids, afinidad_seg, individual, popularidad, alpha):
    """score = alpha * individual + (1 - alpha) * segment.  If there is no history, segment only."""
    base = matriz_segmento(user_ids, afinidad_seg, popularidad)
    ind = individual.reindex(index=user_ids, columns=ALL_CATS).values   # NaN where there is no history
    hay_historial = ~np.isnan(ind)
    resultado = base.copy()
    resultado[hay_historial] = alpha * ind[hay_historial] + (1 - alpha) * base[hay_historial]
    return resultado


def matriz_cluster(user_ids, cluster_aff, popularidad):
    """For each user, the affinity of their demographic cluster (or popularity)."""
    filas = []
    for uid in user_ids:
        c = usuario_cluster.get(uid, -1)
        if c in cluster_aff.index:
            filas.append(cluster_aff.loc[c].values)
        else:
            filas.append(popularidad)
    return np.array(filas)


# --- ranking metrics ---
def dcg(relevancias):
    total = 0.0
    for i, rel in enumerate(relevancias):
        total += rel / np.log2(i + 2)
    return total


def ndcg(orden, clicadas, k):
    relevancias = [1 if cat in clicadas else 0 for cat in orden[:k]]
    ideal = dcg([1] * min(len(clicadas), k))
    return dcg(relevancias) / ideal if ideal > 0 else 0.0


def average_precision(orden, clicadas, k):
    aciertos = 0
    suma = 0.0
    for i, cat in enumerate(orden[:k]):
        if cat in clicadas:
            aciertos += 1
            suma += aciertos / (i + 1)
    return suma / min(len(clicadas), k) if clicadas else 0.0


def categorias_clicadas(split):
    """For each user, the set of categories they clicked in that period."""
    solo_clicks = split[split["target"] == 1]
    return solo_clicks.groupby("id_user")["product_new"].apply(set)


def evaluar(matriz_scores, user_ids, verdad):
    """Compute HR@5, HR@10, NDCG@10 and MAP@10 for a list of users."""
    hr5, hr10, ndcg_list, map_list = [], [], [], []
    for i, uid in enumerate(user_ids):
        orden_idx = np.argsort(-matriz_scores[i])           # categories from highest to lowest score
        orden = [ALL_CATS[j] for j in orden_idx]
        clicadas = verdad[uid]
        hr5.append(1 if len(clicadas & set(orden[:5])) > 0 else 0)
        hr10.append(1 if len(clicadas & set(orden[:10])) > 0 else 0)
        ndcg_list.append(ndcg(orden, clicadas, 10))
        map_list.append(average_precision(orden, clicadas, 10))
    return {"HR5": np.array(hr5), "HR10": np.array(hr10),
            "NDCG": np.array(ndcg_list), "MAP": np.array(map_list)}


print("Functions ready.")

## 5 · Select α on validation (using HR@5)

In [ ]:
verdad_valid = categorias_clicadas(valid)
usuarios_valid = list(verdad_valid.index)

mejor_alpha = None
mejor_hr5 = -1
for alpha in [0.0, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 1.0]:
    scores = matriz_hibrida(usuarios_valid, afinidad_segmento, individual, popularidad, alpha)
    hr5 = evaluar(scores, usuarios_valid, verdad_valid)["HR5"].mean()
    print(f"  alpha={alpha}  ->  HR@5 validation = {hr5:.4f}")
    if hr5 > mejor_hr5:
        mejor_hr5 = hr5
        mejor_alpha = alpha

ALPHA = mejor_alpha
print(f"\nBest alpha: {ALPHA}  (HR@5 validation = {mejor_hr5:.4f})")

## 5.bis · Leak-free segment (correction C1)

The M1 segment from `06` is computed using the user's entire history, including the period used here
as test. This introduces temporal leakage: cluster membership already incorporates the future
behaviour we are trying to predict, which inflates HR@5.

For a leak-free evaluation, the segment is rebuilt using train events only, replicating the
`06` pipeline (`log1p` → `StandardScaler` → `PCA≥90%` → `KMeans k=12`), fitted on train alone. The
cluster→category affinity is likewise computed on train only. In production (section 7) the official
`06` segment with the full known history is used, which does not constitute leakage: at inference time
it is legitimate to use everything observed about the user.

In [ ]:
# ── Rebuilding the M1 segment WITHOUT TEMPORAL LEAKAGE (correction C1) ──
# The 06 segment is computed using the ENTIRE history, including the test period. For a leak-free
# evaluation we rebuild it using ONLY train events, replicating the 06 pipeline
# (log1p on counts -> StandardScaler -> PCA≥90% var -> KMeans k=12), fitted on train only.

SECTORES_06 = ["seguros", "energía", "teleco", "motor", "servicios",
               "hogar", "servicios legales", "salud y belleza", "finanzas"]
LOG_COLS_06 = ["n_eventos", "n_clicks", "n_opens", "n_productos"]
K_SEG = 12  # same k as the final M1 segment of 06 (cluster_km)


def _colname(s):
    return s.strip().lower().replace(" ", "_").replace("í", "i").replace("é", "e")


def construir_features_comportamiento(ev):
    """Replicate the 3 feature blocks of 06 (activity + sector + product) on `ev`."""
    ev = ev.copy()
    ev["es_click"] = ev["target"]
    ev["es_open"] = 1 - ev["target"]

    # Block 1 — general activity
    fref = ev["timestamp"].max()
    agg = ev.groupby("id_user").agg(
        n_eventos=("target", "count"), n_clicks=("es_click", "sum"), n_opens=("es_open", "sum"),
        n_productos=("id_product", "nunique"), n_sectores=("sector", "nunique"),
        ultima=("timestamp", "max"),
    ).reset_index()
    agg["recencia_dias"] = (fref - agg["ultima"]).dt.days
    agg = agg.drop(columns="ultima")
    agg["click_rate"] = agg["n_clicks"] / agg["n_eventos"]

    # Block 2 — sector preference (frequency + click_rate per sector)
    sec = ev[ev["sector"].isin(SECTORES_06)]
    sec_ev = sec.groupby(["id_user", "sector"]).size().unstack(fill_value=0)
    sec_freq = sec_ev.div(sec_ev.sum(axis=1), axis=0)
    sec_freq.columns = ["sec_" + _colname(c) for c in sec_freq.columns]
    sec_cl = sec[sec["target"] == 1].groupby(["id_user", "sector"]).size().unstack(fill_value=0)
    sec_cr = sec_cl.div(sec_ev.replace(0, np.nan)).fillna(0)
    sec_cr.columns = ["cr_" + _colname(c) for c in sec_cr.columns]
    df_sec = sec_freq.join(sec_cr, how="outer").fillna(0).reset_index()

    # Block 3 — product category preference (top-20 of the set itself + 'other')
    top = ev["product_new"].value_counts(normalize=True).head(20).index.tolist()
    ev["prod_bucket"] = ev["product_new"].where(ev["product_new"].isin(top), other="other")
    pc = ev.groupby(["id_user", "prod_bucket"]).size().unstack(fill_value=0)
    pf = pc.div(pc.sum(axis=1), axis=0)
    pf.columns = ["prod_" + c for c in pf.columns]
    df_prod = pf.reset_index()

    return agg.merge(df_sec, on="id_user", how="left").merge(df_prod, on="id_user", how="left").fillna(0)


def segmento_sin_fuga(train_ev, k=K_SEG):
    """Fit the 06 pipeline on train ONLY and return the per-user segment assignment."""
    feat = construir_features_comportamiento(train_ev)
    cols = [c for c in feat.columns if c != "id_user"]
    X = feat[cols].values.astype(float).copy()
    log_idx = [cols.index(c) for c in LOG_COLS_06 if c in cols]
    X[:, log_idx] = np.log1p(X[:, log_idx])
    Xs = StandardScaler().fit_transform(X)
    var = np.cumsum(PCA(random_state=42).fit(Xs).explained_variance_ratio_)
    n_pca = int(np.argmax(var >= 0.90)) + 1
    Xp = PCA(n_components=n_pca, random_state=42).fit_transform(Xs)
    labels = KMeans(n_clusters=k, random_state=42, n_init=20).fit_predict(Xp)
    return pd.Series(labels, index=feat["id_user"], name="seg_lf"), n_pca


usuario_segmento_lf, n_pca_lf = segmento_sin_fuga(train)

# Affinity segmento_sin_fuga -> category, computed on train ONLY (leak-free)
tr_lf = train.copy()
tr_lf["seg_lf"] = tr_lf["id_user"].map(usuario_segmento_lf)
agg_lf = (tr_lf.dropna(subset=["seg_lf"]).groupby(["seg_lf", "product_new"])["target"]
          .agg(n="count", c="sum"))
agg_lf["tasa"] = agg_lf["c"] / agg_lf["n"]
mat_lf = (agg_lf.reset_index().pivot(index="seg_lf", columns="product_new", values="tasa")
          .reindex(columns=ALL_CATS).fillna(0))
afinidad_lf = mat_lf.div(mat_lf.sum(axis=1).replace(0, 1), axis=0)


def matriz_segmento_lf(user_ids):
    """Affinity row of the user's LEAK-FREE segment (or popularity if they have no train)."""
    filas = []
    for uid in user_ids:
        s = usuario_segmento_lf.get(uid)
        filas.append(afinidad_lf.loc[s].values if s in afinidad_lf.index else popularidad)
    return np.array(filas)


print(f"Leak-free segment: KMeans k={K_SEG} on {n_pca_lf} PCA comp. (train only).")
print(f"Users with train segment: {usuario_segmento_lf.notna().sum():,}")

## 6 · Final evaluation on test (with statistical tests)

In [ ]:
verdad_test = categorias_clicadas(test)
usuarios_test = list(verdad_test.index)

NOMBRE_M3 = "Recomendador por cluster (M1)"

# Strategies evaluated on the SAME test users.
# The final recommender serves the CLUSTER top-5 (M1 segment, LEAK-FREE); it includes no
# individual term because validation (section 5) gave α=0.
# As a reference we add the same model with the 06 segment (WITH leakage) to measure the inflation.
resultados = {}
resultados["Popularidad"] = evaluar(
    matriz_segmento(usuarios_test, afinidad_segmento, popularidad) * 0 + popularidad, usuarios_test, verdad_test)
resultados["Cluster demo (05)"] = evaluar(
    matriz_cluster(usuarios_test, cluster_aff, popularidad), usuarios_test, verdad_test)
resultados[NOMBRE_M3] = evaluar(
    matriz_segmento_lf(usuarios_test), usuarios_test, verdad_test)
resultados["(ref.) segmento 06 con fuga"] = evaluar(
    matriz_segmento(usuarios_test, afinidad_segmento, popularidad), usuarios_test, verdad_test)

print(f"Users evaluated (with a click in test): {len(usuarios_test)}\n")
print(f"{'Method':<30}{'HR@5':>8}{'HR@10':>8}{'NDCG@10':>9}{'MAP@10':>8}")
for nombre, m in resultados.items():
    print(f"{nombre:<30}{m['HR5'].mean():>8.4f}{m['HR10'].mean():>8.4f}{m['NDCG'].mean():>9.4f}{m['MAP'].mean():>8.4f}")

inflacion = resultados["(ref.) segmento 06 con fuga"]["HR5"].mean() - resultados[NOMBRE_M3]["HR5"].mean()
print(f"\nHR@5 inflation due to the segment's temporal leakage: +{inflacion:.4f}")

In [ ]:
# Contrast of the cluster recommender's HR@5 (leak-free) against the baselines -> McNemar test
aciertos_m3 = resultados[NOMBRE_M3]["HR5"]
print("McNemar HR@5 (recomendador por cluster vs baseline):")
for nombre in ["Popularidad", "Cluster demo (05)"]:
    aciertos_base = resultados[nombre]["HR5"]
    # users where the recommender hits and the baseline does not (and vice versa)
    cluster_gana = int(((aciertos_m3 == 1) & (aciertos_base == 0)).sum())
    baseline_gana = int(((aciertos_m3 == 0) & (aciertos_base == 1)).sum())
    tabla = [[0, baseline_gana], [cluster_gana, 0]]
    p_valor = mcnemar(tabla, exact=False, correction=True).pvalue
    if p_valor < 0.05:
        veredicto = "el recomendador MEJORA" if cluster_gana > baseline_gana else "el BASELINE es mejor"
    else:
        veredicto = "sin diferencia significativa"
    print(f"  vs {nombre:<18} cluster gana en {cluster_gana}, baseline gana en {baseline_gana}  p={p_valor:.3g}  -> {veredicto}")

# Wilcoxon on the per-user NDCG (cluster vs popularity)
ndcg_m3 = resultados[NOMBRE_M3]["NDCG"]
ndcg_popularidad = resultados["Popularidad"]["NDCG"]
try:
    _, p_wilcoxon = wilcoxon(ndcg_m3, ndcg_popularidad)
    direccion = "cluster > pop" if ndcg_m3.mean() > ndcg_popularidad.mean() else "pop > cluster"
    print(f"\nWilcoxon NDCG@10 (cluster vs popularidad): p={p_wilcoxon:.3g}  ({direccion})")
except ValueError as e:
    print("Wilcoxon no aplicable:", e)

# Confidence interval for the recommender's HR@5 (bootstrap)
rng = np.random.RandomState(42)
medias_bootstrap = []
for _ in range(2000):
    muestra = rng.randint(0, len(aciertos_m3), len(aciertos_m3))
    medias_bootstrap.append(aciertos_m3[muestra].mean())
lo = np.percentile(medias_bootstrap, 2.5)
hi = np.percentile(medias_bootstrap, 97.5)
print(f"HR@5 recomendador por cluster: {aciertos_m3.mean():.4f}   IC95% [{lo:.4f}, {hi:.4f}]")
print(f"HR@5 popularidad (baseline)  : {resultados['Popularidad']['HR5'].mean():.4f}")

In [ ]:
# Comparative chart
metricas = ["HR5", "HR10", "NDCG", "MAP"]
etiquetas = ["HR@5", "HR@10", "NDCG@10", "MAP@10"]
posiciones = np.arange(len(metricas))
ancho = 0.25
centro = (len(resultados) - 1) / 2

fig, ax = plt.subplots(figsize=(10, 4))
for i, (nombre, m) in enumerate(resultados.items()):
    alturas = [m[met].mean() for met in metricas]
    ax.bar(posiciones + (i - centro) * ancho, alturas, ancho, label=nombre)
ax.set_xticks(posiciones)
ax.set_xticklabels(etiquetas)
ax.set_ylabel("Score")
ax.set_title("Recomendador por cluster (M1) vs baselines (test)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 6.bis · Statistical comparison (3 methods)

The three servable methods (popularity, demographic cluster and cluster recommender M1) are compared
using the following tests:

1. Mean ± deviation across users, to reflect dispersion as well as the average.
2. Friedman (global non-parametric test): determines whether any difference exists between the three methods.
   Each user is a block and the methods are ranked by their NDCG@10.
3. Post-hoc paired Wilcoxon on the 3 pairs, with Holm correction for multiple comparisons
   (controls the family-wise error).
4. McNemar of the recommender against each baseline on HR@5 (paired binary), also with Holm.

In [ ]:
from itertools import combinations

from scipy.stats import friedmanchisquare, rankdata
from statsmodels.stats.multitest import multipletests


def rango_biserial(a, b):
    """Paired rank-biserial correlation = effect size of the Wilcoxon, in [-1, 1]
    (~0.1 small, ~0.3 medium, ~0.5 large)."""
    d = np.asarray(a, float) - np.asarray(b, float)
    d = d[d != 0]
    if d.size == 0:
        return 0.0
    r = rankdata(np.abs(d))
    return (r[d > 0].sum() - r[d < 0].sum()) / r.sum()

# REAL methods to compare (we exclude the "with leakage" reference, which is not a servable model)
metodos = ["Popularidad", "Cluster demo (05)", NOMBRE_M3]

# ── Dispersion: mean ± sd across users (not just the mean) ──
print("Mean ± sd across test users:")
print(f"{'Method':<30}{'HR@5':>18}{'NDCG@10':>18}")
for nombre in metodos:
    h, n = resultados[nombre]["HR5"], resultados[nombre]["NDCG"]
    print(f"{nombre:<30}{h.mean():>10.4f} ± {h.std():.3f}{n.mean():>9.4f} ± {n.std():.3f}")

# ── 1) Friedman: global difference between the 3 methods (NDCG@10 per user) ──
ndcgs = [resultados[m]["NDCG"] for m in metodos]   # aligned: same users, same order
chi2, p_fried = friedmanchisquare(*ndcgs)
print(f"\nFriedman (NDCG@10, 3 methods, {len(usuarios_test)} users): chi2={chi2:.1f}  p={p_fried:.3g}")
W_kendall = chi2 / (len(usuarios_test) * (len(metodos) - 1))   # Friedman effect size (0-1)
print(f"  Global effect size (Kendall's W): {W_kendall:.3f}  (~0.1 small, ~0.3 medium, ~0.5 large)")

# ── 2) Paired post-hoc: Wilcoxon on NDCG@10 for the 3 pairs + Holm correction ──
pares = list(combinations(metodos, 2))
pvals, etiquetas, signos, efectos = [], [], [], []
for a, b in pares:
    _, p = wilcoxon(resultados[a]["NDCG"], resultados[b]["NDCG"])
    pvals.append(p)
    etiquetas.append(f"{a} vs {b}")
    signos.append(f"({a if resultados[a]['NDCG'].mean() > resultados[b]['NDCG'].mean() else b} mayor)")
    efectos.append(rango_biserial(resultados[a]["NDCG"], resultados[b]["NDCG"]))
rech, p_holm, _, _ = multipletests(pvals, alpha=0.05, method="holm")
print("\nPost-hoc Wilcoxon NDCG@10 + Holm:")
for et, sg, p0, ph, rh, ef in zip(etiquetas, signos, pvals, p_holm, rech, efectos):
    print(f"  {et:<46} p={p0:.2e}  p_holm={ph:.2e}  {'SIG' if rh else 'n.s.'}  |r|={abs(ef):.3f} {sg}")

# ── 3) McNemar HR@5 of the recommender vs each baseline + Holm correction ──
pmc, etmc, dirmc = [], [], []
for nombre in ["Popularidad", "Cluster demo (05)"]:
    ab, bb = resultados[NOMBRE_M3]["HR5"], resultados[nombre]["HR5"]
    b01, b10 = int(((ab == 0) & (bb == 1)).sum()), int(((ab == 1) & (bb == 0)).sum())
    pmc.append(mcnemar([[0, b01], [b10, 0]], exact=False, correction=True).pvalue)
    etmc.append(f"cluster vs {nombre}")
    dirmc.append("cluster mejor" if b10 > b01 else "baseline mejor")
rech_mc, p_holm_mc, _, _ = multipletests(pmc, alpha=0.05, method="holm")
print("\nMcNemar HR@5 (recomendador vs baseline) + Holm:")
for et, p0, ph, rh, dr in zip(etmc, pmc, p_holm_mc, rech_mc, dirmc):
    print(f"  {et:<40} p={p0:.2e}  p_holm={ph:.2e}  {'SIG' if rh else 'n.s.'} ({dr})")

## 7 · Final recommendations per cluster

The recommender computes the top-5 at cluster level (M1 segment) and serves it to all its members.
The signals are rebuilt with all the data (at inference time the full known history of the user is
used, which is legitimate) and two tables are saved:

1. `cluster_recommendations.csv` — the top-5 of each cluster.
2. `recommendations.csv` — the top-5 of each user, obtained via a `user → their cluster → top-5` *lookup*.
   It preserves the per-user interface for the API; the content is that of their cluster.

In [ ]:
# Signals with all the data (at inference time the full known history is used)
afinidad_f, individual_f, popularidad_f, cluster_f = construir_senales(df)
todos_usuarios = sorted(df["id_user"].unique())

# Popularity top-5 (fallback for users with no assigned segment)
pop_top5 = [ALL_CATS[j] for j in np.argsort(-popularidad_f)[:5]]


def top5_de_scores(scores):
    """Return (categories, scores) of the top-5 from a vector of per-category scores."""
    mejores = np.argsort(-scores)[:5]
    return [ALL_CATS[j] for j in mejores], [round(float(scores[j]), 6) for j in mejores]


# 1) PER-CLUSTER table (M1 segment): the top-5 served to all members of the segment
filas_cluster = []
for seg in ALL_SEG:
    scores = afinidad_f.loc[seg].values if seg in afinidad_f.index else popularidad_f
    recs, scs = top5_de_scores(scores)
    fila = {"cluster_id": seg, "fuente": "segmento_m1"}
    for puesto in range(1, 6):
        fila[f"rec_{puesto}"] = recs[puesto - 1]
        fila[f"score_{puesto}"] = scs[puesto - 1]
    filas_cluster.append(fila)

cluster_recs = pd.DataFrame(filas_cluster)
cluster_recs.to_csv(PROCESSED_PATH / "cluster_recommendations.csv", index=False)
print("Saved: cluster_recommendations.csv", cluster_recs.shape, "(top-5 per cluster)")

# Dictionaries cluster -> top-5 (for the per-user lookup)
top5_por_cluster = {r["cluster_id"]: [r[f"rec_{i}"] for i in range(1, 6)] for _, r in cluster_recs.iterrows()}
score5_por_cluster = {r["cluster_id"]: [r[f"score_{i}"] for i in range(1, 6)] for _, r in cluster_recs.iterrows()}

# 2) PER USER: lookup user -> their cluster -> cluster top-5 (interface compatibility)
filas = []
for uid in todos_usuarios:
    seg = usuario_segmento.get(uid)
    if seg in top5_por_cluster:
        recs, scs = top5_por_cluster[seg], score5_por_cluster[seg]
    else:
        recs, scs = pop_top5, [0.0] * 5   # user with no segment -> popularity
    fila = {"id_user": uid}
    for puesto in range(1, 6):
        fila[f"rec_{puesto}"] = recs[puesto - 1]
        fila[f"score_{puesto}"] = scs[puesto - 1]
    filas.append(fila)

recomendaciones = pd.DataFrame(filas)
recomendaciones.to_csv(PROCESSED_PATH / "recommendations.csv", index=False)
print("Saved: recommendations.csv", recomendaciones.shape, "(via user→cluster lookup)")
display(cluster_recs.head())
recomendaciones.head()

## 8 · Cold start (new user)

A new user has no segment (history is required). We assign them their **demographic cluster** and
give them the most affine categories of that cluster.

In [ ]:
def recomendar_nuevo(demo_cluster, k=5):
    if demo_cluster in cluster_f.index:
        scores = cluster_f.loc[demo_cluster].values
    else:
        scores = popularidad_f
    mejores = np.argsort(-scores)[:k]
    return pd.DataFrame({
        "puesto": range(1, k + 1),
        "product_new": [ALL_CATS[j] for j in mejores],
        "score": np.round(scores[mejores], 4),
    })


print("Cold-start example (demographic cluster = 3):")
recomendar_nuevo(3)

## Summary of decisions (M3)

| # | Decision | Why |
|---|----------|---------|
| **Per cluster** | the top-5 is computed at cluster level (M1 segment) and served to all its members | Individual history is sparse (median ≈1 event); the useful signal is the cluster's |
| No individual term | α=0 on validation → the `individual` component is removed from production | Blending `α·individual + (1−α)·cluster` does not help; the validation optimum is α=0 |
| Cold start | new user → their **demographic cluster** (05); no cluster → popularity | The cold start has no history, but it does have a demographic profile |
| Temporal split | train / validation / test by date; α chosen on validation | No temporal leakage; α is not tuned on test |
| **Leak-free evaluation (C1)** | the segment is rebuilt with train ONLY (5.bis); production uses the 06 segment with the full history | Prevents cluster membership from incorporating the future. The leakage inflated HR@5 by ≈ +0.10 |
| Tests | HR/NDCG/MAP + McNemar + Wilcoxon + bootstrap | Comparison with statistical significance |

**Conclusion:** without leakage, the cluster recommender does not beat popularity (HR@5 ≈ 0.59 vs 0.63;
McNemar favours popularity). Its value is operational (interpretable segments, cold start), not a
predictive improvement over a popularity baseline.

**Outputs:**
- `data/processed/cluster_recommendations.csv` (`cluster_id`, `fuente`, `rec_1..5`, `score_1..5`) — recommendation per cluster.
- `data/processed/recommendations.csv` (`id_user`, `rec_1..5`, `score_1..5`) — per user via `user → cluster` *lookup*.